# Conformal prediction with RAPS

We have extended and adapted the code from the original RAPS Colab. To preserve anonymity, we provide the code in this notebook. However, we have run all this code in a Colab environment for our experiments. Below a few references to RAPS:
* [Website](https://people.eecs.berkeley.edu/~angelopoulos/blog/posts/conformal-classification/)
* [Paper](https://people.eecs.berkeley.edu/~angelopoulos/publications/downloads/conformal-classification.pdf)
* [Original RAPS Colab](https://colab.research.google.com/github/aangelopoulos/conformal_classification/blob/master/example.ipynb)

## Exploring `RAPS`
This is a colab that shows you how to generate predictive sets guaranteed to contain the true class label with a probability you specify.

This technique was proposed in our work [Uncertainty Sets for Image Classifiers using Conformal Prediction](https://arxiv.org/abs/).

## Setting up the experiments

**Make sure you're using a GPU** by setting "Runtime/Change runtime type/Hardware accelerator/" to "GPU" in the colab menu above.  

First, let's deal with imports, loading the pretrained model, and other boilerplate.

In [ ]:
!rm -r conformal_classification
!git clone https://github.com/aangelopoulos/conformal_classification.git
import os, sys, inspect
sys.path.insert(1, os.path.join(sys.path[0], './conformal_classification/'))

from conformal import *
from utils import *

# Import other standard packages
import argparse
import torch
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import time
import torch.backends.cudnn as cudnn
import random

# Fix the random seed for reproducibility (you can change this, of course)
seed=0
np.random.seed(seed=seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
random.seed(seed)

# Normalization from torchvision repo
transform = transforms.Compose([
                transforms.Resize(256),
                transforms.CenterCrop(224),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                      std= [0.229, 0.224, 0.225])
            ])

cudnn.benchmark = True
batch_size = 128

# Get your model
model = torchvision.models.resnet152(pretrained=True,progress=True).cuda()
_ = model.eval()

Now download and extract ImageNet-Val. You only need to execute this cell once and it takes less than 5 minutes to run on my system.

In [ ]:
!wget -nv -O imagenet_val.tar.gz -L https://berkeley.box.com/shared/static/pouthcomrvxw9hj64oxhacjvqdw3ihlp.gz
!mkdir imagenet_val
!tar -xf imagenet_val.tar.gz -C ./imagenet_val/
!mv imagenet_val/scratch/group/ilsvrc/val/* imagenet_val/
!rm -r imagenet_val/scratch

## Get conformal calibration and validation dataset

Now get the conformal calibration set and the validation dataset. To ensure there is a sufficient number of labels in the subset, we first randomly sample a number of labels

In [ ]:
import random
import numpy as np
from sklearn.model_selection import train_test_split

# We set a seed to make our experiments reproducible
seed = 42
np.random.seed(seed)
random.seed(seed)

dataset = torchvision.datasets.ImageFolder('./imagenet_val/', transform)

For the hierarchical conformal prediction, it is optimal to have as many instances per class. Instead of randomly selecting n samples, we randomly select a number of labels, that determines the samples that are chosen.

In [ ]:
from collections import Counter
from torch.utils.data import Subset

def select_subset(ds, subset_size):
  """ Select data, the aim is to have all 50 instance labels per class """
  nb_labels = subset_size // 50
  labels = list(Counter(ds.targets).keys())
  random.seed(seed)
  sampled_labels = set(random.sample(labels, nb_labels))
  indices = np.where(np.isin(ds.targets, list(sampled_labels)))[0]
  return Subset(ds, indices), indices

imagenet_subset, indices = select_subset(ds=dataset, subset_size=10000)

In [ ]:
calib_indices, val_indices = train_test_split(indices, test_size=0.8, random_state=42, stratify=[dataset.targets[i] for i in indices])

In [ ]:
imagenet_calib_data = Subset(dataset, calib_indices)
imagenet_val_data = Subset(dataset, val_indices)

In [ ]:
# Initialize loaders
calib_loader = torch.utils.data.DataLoader(imagenet_calib_data, batch_size=batch_size, shuffle=True, pin_memory=True)
val_loader = torch.utils.data.DataLoader(imagenet_val_data, batch_size=batch_size, shuffle=True, pin_memory=True)

## Conformalizing the model

This this the key step, where you perform Platt scaling and then wrap the model with `RAPS`.

 *(Did you catch the joke?)*

You should experiment with different values of `alpha` and `lamda_criterion`=`'size'` or `'adaptiveness'` to see how the sets change.

# Conformalize model
ALPHA = 0.05
cmodel = ConformalModel(model, calib_loader, alpha=ALPHA, lamda_criterion='size')

## 1-alpha predictive sets

**Congratulations!** You can now output `1-alpha` predictive sets.


Now you can validate the coverage of your conformal model on the validation set with this utility function.

Running averages of coverage and size are in parentheses.

**Feel free to stop execution of this cell at any time**; otherwise it will run until `N=50,000-num_calib`, since that's the size of the validation set.

In [ ]:
top1, top5, coverage, size = validate(val_loader, cmodel, print_bool=True)

## Visualise examples

That's it! You're ready to use RAPS.

As a final visualization, running the next cell will just sample random 5 images and show their RAPS sets. Note that you can avoid sets of size zero by setting `randomized=False`.

It's interesting to see which images the model thinks are more difficult.

In [ ]:
num_images = 20
torch.manual_seed(23)
num_calib = 2000
explore_data, _ = torch.utils.data.random_split(imagenet_val_data, [num_images, 10000-num_calib-num_images])

import pdb
import json
import ast
with open('imagenet_label_to_wordnet_synset.txt', 'r') as f:
    labels = f.read()
    st = ast.literal_eval(labels)
    labeldict = {key: value['label'].split(',')[0] for key, value in st.items()}

mosaiclist = []
sets = []
labels = []

for data in explore_data:
  img, label = data
  scores, set = cmodel(img.view(1,3,224,224).cuda())
  print("set:", set)
  for i in set[0]:
    if i == 638:
      print("This is the true label:", label)
  unnormalized_img = (img * torch.Tensor([0.229, 0.224, 0.225]).view(-1,1,1))+torch.Tensor([0.485, 0.456, 0.406]).view(-1,1,1)

  set = [labeldict[s] for s in set[0]]
  sets = sets + [set]
  labels = labels + [labeldict[label]]
  mosaiclist = mosaiclist + [unnormalized_img]

grid = torchvision.utils.make_grid(mosaiclist)

fig, ax = plt.subplots(figsize=(min(num_images,9)*5, np.floor(num_images/9+1)*5))
ax.imshow(grid.permute(1,2,0), interpolation='nearest')
ax.get_xaxis().set_visible(False)
ax.get_yaxis().set_visible(False)
plt.tight_layout()

for i in range(len(mosaiclist)):
  print(f"Image {i} has label \'{labels[i]}\', and the predictive set is {sets[i]}.")

## Save data 

We save both the calibration and test data, to ensure we can use the same data for the hierarchical conformation prediction model.

In [ ]:
from PIL import Image
from tqdm import tqdm
import ast

In [ ]:
def save_image(tensor, save_path):
  """ Save tensor into .png """
  np_array = tensor.permute(1, 2, 0).numpy()  # Change from (C, H, W) to (H, W, C)
  image = Image.fromarray((np_array * 255).astype(np.uint8))
  image.save(save_path)

In [ ]:
def get_all_ps():
  """ Get all prediction sets (both for the val data and calib data) """
  with open('imagenet_label_to_wordnet_synset.txt', 'r') as f:
    labels = f.read()
    st = ast.literal_eval(labels)
    labeldict = {key: value['label'].split(',')[0] for key, value in st.items()}

  mosaiclist, sets, labels, all_scores = [], [], [], []

  for data in tqdm(imagenet_subset):
    #print(f"\rProcessing image {i+1}/{len(explore_data)}...", end="")
    img, label = data
    scores, set = cmodel(img.view(1,3,224,224).cuda())
    unnormalized_img = (img * torch.Tensor([0.229, 0.224, 0.225]).view(-1,1,1))+torch.Tensor([0.485, 0.456, 0.406]).view(-1,1,1)

    # set = [labeldict[s] for s in set[0]]
    sets = sets + [set]
    # labels = labels + [labeldict[label]]
    labels = labels + [label]
    mosaiclist = mosaiclist + [unnormalized_img]
    all_scores.append(scores.cpu().detach().numpy())
  return mosaiclist, labels, sets, all_scores

In [ ]:
mosaiclist, labels, sets, all_scores = get_all_ps()

In [ ]:
import subprocess

for directory in ["data", f"data/{str(ALPHA)}", f"data/{str(ALPHA)}/images10k", f"data/{str(ALPHA)}/images45k"]:
  if not os.path.exists(directory):
    subprocess.call(f"mkdir {directory}", shell=True)

In [ ]:
import json
from tqdm import tqdm

for index, tensor in tqdm(enumerate(mosaiclist), total=len(mosaiclist)):
  save_image(tensor=tensor, save_path=f"./data/{str(ALPHA)}/images10k/{index}.png")

results = {index: {"label": label, "set": sets[index][0].tolist()} for index, label in enumerate(labels)}
with open(f'./data/{str(ALPHA)}/results10k_number.json', 'w') as json_file:
    json.dump(results, json_file, indent=4)

conformal_scores = np.vstack(all_scores)
save_path = f'./data/{str(ALPHA)}/scores.npy'
np.save(save_path, conformal_scores)

In [ ]:
!zip -r data/{ALPHA}.zip data/{ALPHA}

In [ ]:
from google.colab import files
files.download(f"data/{ALPHA}.zip")